<h1>Q1</h1>

In [2]:
pip install snorkel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 1.8 MB/s eta 0:00:00


In [5]:

# ====================================================================
# Q4: Sequential Training with W&B (CIFAR 10/100) (COMPLETED)
# ====================================================================
print("\n--- Starting Q4: Sequential CIFAR Training ---")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS_PER_STAGE = 100
BATCH_SIZE = 128

# Simple CNN Model
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv_stack = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2), # 32x16x16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2)  # 64x8x8
        )
        self.fc = nn.Linear(64 * 8 * 8, num_classes)

    def forward(self, x):
        x = self.conv_stack(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

    def update_num_classes(self, new_num_classes):
        in_features = self.fc.in_features
        self.fc = nn.Linear(in_features, new_num_classes)
        return self # Return self to allow chaining .to(device)

# Data Transformations and Loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
trainloader_10 = torch.utils.data.DataLoader(
    torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform),
    batch_size=BATCH_SIZE, shuffle=True
)
trainloader_100 = torch.utils.data.DataLoader(
    torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform),
    batch_size=BATCH_SIZE, shuffle=True
)

# Training Function
def train_model_stage(model, dataloader, criterion, optimizer, num_epochs, task_name, log_offset):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(dataloader)
        epoch_acc = 100 * correct / total

        wandb.log({
            "epoch": epoch + 1 + log_offset,
            f"{task_name}_loss": epoch_loss,
            f"{task_name}_accuracy": epoch_acc,
        })
        print(f"Epoch {epoch+1+log_offset} ({task_name}): Loss={epoch_loss:.4f}, Acc={epoch_acc:.2f}%")
    return model

# --- Experiment (a): CIFAR 100 then CIFAR 10 ---
wandb.init(project="Q4-cifar-experiments", name="CIFAR100_then_10")

# Stage 1: C100 (100 classes)
model_a = SimpleCNN(num_classes=100).to(device)
criterion_a = nn.CrossEntropyLoss()
optimizer_a = optim.Adam(model_a.parameters(), lr=0.001)

model_a = train_model_stage(model_a, trainloader_100, criterion_a, optimizer_a,
                            num_epochs=EPOCHS_PER_STAGE, task_name="C100_S1", log_offset=0)

# Stage 2: C10 (10 classes)
model_a = model_a.update_num_classes(10).to(device)
optimizer_a = optim.Adam(model_a.parameters(), lr=0.001) # Reset optimizer
model_a = train_model_stage(model_a, trainloader_10, criterion_a, optimizer_a,
                            num_epochs=EPOCHS_PER_STAGE, task_name="C10_S2", log_offset=EPOCHS_PER_STAGE)

wandb.finish()


# --- Experiment (b): CIFAR 10 then CIFAR 100 ---
wandb.init(project="Q4-cifar-experiments", name="CIFAR10_then_100")

# Stage 1: C10 (10 classes)
model_b = SimpleCNN(num_classes=10).to(device)
criterion_b = nn.CrossEntropyLoss()
optimizer_b = optim.Adam(model_b.parameters(), lr=0.001)

model_b = train_model_stage(model_b, trainloader_10, criterion_b, optimizer_b,
                            num_epochs=EPOCHS_PER_STAGE, task_name="C10_S1", log_offset=0)

# Stage 2: C100 (100 classes)
model_b = model_b.update_num_classes(100).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=0.001) # Reset optimizer
model_b = train_model_stage(model_b, trainloader_100, criterion_b, optimizer_b,
                            num_epochs=EPOCHS_PER_STAGE, task_name="C100_S2", log_offset=EPOCHS_PER_STAGE)

wandb.finish()
print("\nQ4 finished. Two sequential runs logged to W&B.")


--- Starting Q4: Sequential CIFAR Training ---


100%|██████████| 170M/170M [00:14<00:00, 11.8MB/s]
100%|██████████| 169M/169M [00:13<00:00, 12.2MB/s]
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


...

Epoch 1 (C100_S1): Loss=3.4556, Acc=19.76%
Epoch 2 (C100_S1): Loss=2.6656, Acc=34.61%
Epoch 3 (C100_S1): Loss=2.2924, Acc=42.63%
Epoch 4 (C100_S1): Loss=2.0162, Acc=48.38%
Epoch 5 (C100_S1): Loss=1.7902, Acc=53.60%
Epoch 6 (C100_S1): Loss=1.5939, Acc=58.23%
Epoch 7 (C100_S1): Loss=1.4103, Acc=62.40%
Epoch 8 (C100_S1): Loss=1.2381, Acc=66.79%
Epoch 9 (C100_S1): Loss=1.0802, Acc=70.70%
Epoch 10 (C100_S1): Loss=0.9468, Acc=74.02%
Epoch 11 (C100_S1): Loss=0.8141, Acc=77.43%
Epoch 12 (C100_S1): Loss=0.7117, Acc=80.18%
Epoch 13 (C100_S1): Loss=0.6045, Acc=82.96%
Epoch 14 (C100_S1): Loss=0.5117, Acc=85.65%
Epoch 15 (C100_S1): Loss=0.4465, Acc=87.45%
Epoch 16 (C100_S1): Loss=0.3781, Acc=89.23%
Epoch 17 (C100_S1): Loss=0.3296, Acc=90.59%
Epoch 18 (C100_S1): Loss=0.2820, Acc=92.01%
Epoch 19 (C100_S1): Loss=0.2577, Acc=92.48%
Epoch 20 (C100_S1): Loss=0.2219, Acc=93.62%
Epoch 21 (C100_S1): Loss=0.1931, Acc=94.47%
Epoch 22 (C100_S1): Loss=0.1815, Acc=94.69%
Epoch 23 (C100_S1): Loss=0.1748, Acc=94.7

C100_S1_accuracy,▁▃▄▄▆▇▇▇████████████████████████████████
C100_S1_loss,█▆▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
C10_S2_accuracy,▁▃▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇██████████████
C10_S2_loss,█▇▇▇▆▅▅▅▅▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▂▂▂▂▂▂▂▂▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇██████
C100_S1_accuracy,98.814
C100_S1_loss,0.03704
C10_S2_accuracy,97.2
C10_S2_loss,0.07929
epoch,200


Epoch 1 (C10_S1): Loss=1.4456, Acc=48.85%
Epoch 2 (C10_S1): Loss=1.0991, Acc=61.69%
Epoch 3 (C10_S1): Loss=0.9608, Acc=66.81%
Epoch 4 (C10_S1): Loss=0.8694, Acc=70.00%
Epoch 5 (C10_S1): Loss=0.8073, Acc=71.92%
Epoch 6 (C10_S1): Loss=0.7553, Acc=74.04%
Epoch 7 (C10_S1): Loss=0.7128, Acc=75.36%
Epoch 8 (C10_S1): Loss=0.6738, Acc=76.81%
Epoch 9 (C10_S1): Loss=0.6370, Acc=78.10%
Epoch 10 (C10_S1): Loss=0.6112, Acc=78.94%
Epoch 11 (C10_S1): Loss=0.5780, Acc=80.10%
Epoch 12 (C10_S1): Loss=0.5546, Acc=80.92%
Epoch 13 (C10_S1): Loss=0.5287, Acc=81.84%
Epoch 14 (C10_S1): Loss=0.5032, Acc=82.58%
Epoch 15 (C10_S1): Loss=0.4804, Acc=83.55%
Epoch 16 (C10_S1): Loss=0.4658, Acc=83.82%
Epoch 17 (C10_S1): Loss=0.4451, Acc=84.59%
Epoch 18 (C10_S1): Loss=0.4298, Acc=85.26%
Epoch 19 (C10_S1): Loss=0.4100, Acc=85.71%
Epoch 20 (C10_S1): Loss=0.3985, Acc=86.18%
Epoch 21 (C10_S1): Loss=0.3826, Acc=86.76%
Epoch 22 (C10_S1): Loss=0.3644, Acc=87.44%
Epoch 23 (C10_S1): Loss=0.3519, Acc=87.71%
Epoch 24 (C10_S1): L

C100_S2_accuracy,▁▄▅█████████████████████████████████████
C100_S2_loss,█▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
C10_S1_accuracy,▁▂▃▃▄▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█████████████████
C10_S1_loss,█▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇██
C100_S2_accuracy,99.466
C100_S2_loss,0.02172
C10_S1_accuracy,98.084
C10_S1_loss,0.05451
epoch,200



Q4 finished. Two sequential runs logged to W&B.


In [ ]:
print("End Q4")